# RAS 2026 — Hybrid RL+SA Solver

**Pipeline:** Greedy → BC (warm-start) → REINFORCE+Entropy (exploration) → SA (exploitation)

**Paper:** *RL-guided Simulated Annealing for Railroad Blocking Plan Optimization*

## 0. Setup

In [ ]:
import os
if not os.path.exists('ras2026'):
    !git clone https://github.com/nepersoned/ras2026.git
os.chdir('ras2026')
print('cwd:', os.getcwd())

In [ ]:
!pip install -q scipy numpy pandas torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/ras_release_v1.1'
print('Exists:', os.path.exists(DRIVE_DATA))

if not os.path.exists('ras_release_v1.1'):
    os.symlink(DRIVE_DATA, 'ras_release_v1.1')
print('Ready:', os.path.exists('ras_release_v1.1/ras_release_v1.1/datasets/l1/node.csv'))

In [ ]:
import sys, math, random, time, json, csv
from pathlib import Path
from collections import defaultdict
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

sys.path.insert(0, '.')
from solver import (
    load_layer, load_od_matrix, Network, Demand, Solution,
    greedy_construct, evaluate, build_json, simulated_annealing,
    COMMODITY_TO_BLOCK_TYPE, DIRECT_ONLY,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
os.makedirs('saved_models', exist_ok=True)

## 1. Environment

In [ ]:
BLOCK_TYPES = ['Manifest', 'Bulk', 'Intermodal', 'Multilevel']
MAX_HUBS    = 50
FEAT_DIM    = 7 + MAX_HUBS * 4
N_ACTIONS   = MAX_HUBS + 2

def load_env(layer, multiplier):
    nodes_df, links_df, demands_scaled, demands_raw, settings = load_layer(layer, multiplier)
    yard_rows = nodes_df[nodes_df['node_type'] == 'yard']
    yard_info = {
        int(r['node_id']): {
            'num_tracks':        float(r.get('num_tracks',        9999) or 9999),
            'handling_capacity': float(r.get('handling_capacity', 1e9)  or 1e9),
            'handling_cost':     float(r.get('handling_cost',     0)    or 0),
        } for _, r in yard_rows.iterrows()
    }
    origin_ids   = set(demands_scaled['origin_yard_id'].astype(int))
    dest_ids     = set(demands_scaled['dest_yard_id'].astype(int))
    all_yard_ids = sorted(origin_ids | dest_ids)
    od_matrix    = load_od_matrix({(o,d) for o in all_yard_ids for d in all_yard_ids if o!=d})
    net = Network(nodes_df, links_df, origin_ids, dest_ids, settings, verbose=True)
    demands = [
        Demand(
            idx=idx, demand_id=int(row['demand_id']),
            commodity_type=str(row['block_type']),
            block_type=COMMODITY_TO_BLOCK_TYPE.get(str(row['block_type']), 'Manifest'),
            origin=int(row['origin_yard_id']), dest=int(row['dest_yard_id']),
            volume=int(row['volume']),
            sp_dist=od_matrix.get(
                (int(row['origin_yard_id']), int(row['dest_yard_id'])),
                net.dist(int(row['origin_yard_id']), int(row['dest_yard_id']))
            ),
        ) for idx, (_, row) in enumerate(demands_scaled.iterrows())
    ]
    return dict(net=net, od_matrix=od_matrix, settings=settings,
                yard_info=yard_info, demands=demands, all_yards=all_yard_ids,
                nodes_df=nodes_df, links_df=links_df, demands_raw=demands_raw)


def ranked_hubs(dem, env):
    if dem.commodity_type in DIRECT_ONLY:
        return []
    scored = []
    for hub in env['all_yards']:
        if hub == dem.origin or hub == dem.dest: continue
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2): continue
        scored.append((d1+d2, hub))
    scored.sort()
    return [h for _, h in scored]


def demand_feature(dem, hubs, env):
    od_d  = env['net'].dist(dem.origin, dem.dest)
    od_ic = env['net'].interchanges(dem.origin, dem.dest)
    bt_oh = [1.0 if COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest') == t
             else 0.0 for t in BLOCK_TYPES]
    base  = [math.log1p(dem.volume),
             math.log1p(od_d if math.isfinite(od_d) else 1e6),
             od_ic / 5.0] + bt_oh
    hub_feats = [0.0] * (MAX_HUBS * 4)
    for j, hub in enumerate(hubs[:MAX_HUBS]):
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2): continue
        ic1 = env['net'].interchanges(dem.origin, hub)
        ic2 = env['net'].interchanges(hub, dem.dest)
        hc  = env['yard_info'].get(hub, {}).get('handling_cost', 0.0)
        hub_feats[j*4]   = math.log1p(d1)
        hub_feats[j*4+1] = math.log1p(d2)
        hub_feats[j*4+2] = (ic1+ic2) / 5.0
        hub_feats[j*4+3] = hc / 500.0
    return base + hub_feats


def action_mask(dem, hubs, env):
    mask = [False] * N_ACTIONS
    if not math.isinf(env['net'].dist(dem.origin, dem.dest)):
        mask[0] = True
    if dem.commodity_type not in DIRECT_ONLY:
        for j, hub in enumerate(hubs[:MAX_HUBS]):
            d1 = env['net'].dist(dem.origin, hub)
            d2 = env['net'].dist(hub, dem.dest)
            if not math.isinf(d1) and not math.isinf(d2):
                mask[j+1] = True
    mask[N_ACTIONS-1] = True
    return mask


def action_to_route(action, dem, hubs):
    if action == 0: return [(dem.origin, dem.dest)]
    if action == N_ACTIONS-1: return None
    hi = action - 1
    return [(dem.origin, hubs[hi]), (hubs[hi], dem.dest)] if hi < len(hubs) else None


def precompute(env):
    data = []
    for dem in env['demands']:
        if dem.volume <= 0:
            data.append(None); continue
        hubs = ranked_hubs(dem, env)
        data.append({'feat': demand_feature(dem, hubs, env),
                     'mask': action_mask(dem, hubs, env),
                     'hubs': hubs})
    return data

## 2. Policy Network

In [ ]:
class BlockingPolicy(nn.Module):
    """MLP policy: 207-dim demand features → N_ACTIONS logits."""
    def __init__(self, feat_dim=FEAT_DIM, n_actions=N_ACTIONS, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, x):
        return self.net(x)

def make_policy():
    return BlockingPolicy().to(DEVICE)

print(f'Policy params: {sum(p.numel() for p in make_policy().parameters()):,}')

## 3. Behavioral Cloning

In [ ]:
def greedy_label(route, hubs):
    if route is None: return N_ACTIONS - 1
    if len(route) == 1: return 0
    hub = route[0][1]
    top = hubs[:MAX_HUBS]
    return top.index(hub) + 1 if hub in top else N_ACTIONS - 1


def pretrain_bc(policy, env, dd, n_epochs=100, lr=1e-3, print_every=25):
    init_sol = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                                env['settings'], env['yard_info'])
    score0, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                         env['settings'], env['yard_info'])

    feats, masks, labels = [], [], []
    for i, dem in enumerate(env['demands']):
        if dd[i] is None: continue
        feats.append(dd[i]['feat'])
        masks.append(dd[i]['mask'])
        labels.append(greedy_label(init_sol.routings[i], dd[i]['hubs']))

    feat_t  = torch.tensor(feats,  dtype=torch.float32, device=DEVICE)
    label_t = torch.tensor(labels, dtype=torch.long,    device=DEVICE)
    mask_t  = torch.tensor(masks,  dtype=torch.bool,    device=DEVICE)

    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    print(f'\n── BC ──  N={len(labels)}  greedy={score0:,.0f}')

    for ep in range(1, n_epochs + 1):
        policy.train()
        logits = policy(feat_t).masked_fill(~mask_t, -1e9)
        loss   = F.cross_entropy(logits, label_t)
        opt.zero_grad(); loss.backward(); opt.step()
        if ep % print_every == 0:
            with torch.no_grad():
                acc = (logits.argmax(1) == label_t).float().mean().item() * 100
            print(f'  ep {ep:3d} | loss={loss.item():.4f} | acc={acc:.1f}%')

    return init_sol, score0

## 4. REINFORCE + Entropy

In [ ]:
@torch.no_grad()
def sample_solution(policy, env, dd, greedy=False, temperature=1.0):
    policy.eval()
    valid_idxs = [i for i,d in enumerate(dd) if d is not None]
    if not valid_idxs: return [None]*len(dd), []

    feats  = torch.tensor([dd[i]['feat'] for i in valid_idxs], dtype=torch.float32, device=DEVICE)
    masks  = torch.tensor([dd[i]['mask'] for i in valid_idxs], dtype=torch.bool,    device=DEVICE)
    logits = policy(feats).masked_fill(~masks, -1e9) / temperature

    actions_valid = logits.argmax(1).tolist() if greedy else Categorical(logits=logits).sample().tolist()

    routings = [None]*len(dd)
    actions  = [N_ACTIONS-1]*len(dd)
    for j, i in enumerate(valid_idxs):
        a = actions_valid[j]
        actions[i]  = a
        routings[i] = action_to_route(a, env['demands'][i], dd[i]['hubs'])
    return routings, actions


def compute_log_probs_entropy(policy, env, dd, actions):
    valid_idxs = [i for i,d in enumerate(dd) if d is not None]
    feats  = torch.tensor([dd[i]['feat'] for i in valid_idxs], dtype=torch.float32, device=DEVICE)
    masks  = torch.tensor([dd[i]['mask'] for i in valid_idxs], dtype=torch.bool,    device=DEVICE)
    act_t  = torch.tensor([actions[i] for i in valid_idxs],    dtype=torch.long,    device=DEVICE)

    logits = policy(feats).masked_fill(~masks, -1e9)
    log_p  = F.log_softmax(logits, dim=1)
    probs  = log_p.exp()

    selected_log_p = log_p.gather(1, act_t.unsqueeze(1)).squeeze(1).sum()
    entropy = -(probs * log_p).sum(1).mean()   # mean entropy over demands
    return selected_log_p, entropy


def train_reinforce(policy, env, dd, n_episodes=500, lr=1e-4,
                    entropy_coef=0.05, baseline_decay=0.95,
                    temperature=1.2, print_every=50):
    opt = torch.optim.Adam(policy.parameters(), lr=lr)

    routings, _ = sample_solution(policy, env, dd, greedy=True)
    sol = Solution(env['demands'], routings)
    baseline, _ = evaluate(sol, env['net'], env['od_matrix'], env['settings'], env['yard_info'])
    best_score  = baseline
    best_state  = deepcopy(policy.state_dict())

    log = []
    print(f'\n── REINFORCE+Entropy ──  init={baseline:,.0f}  T={temperature}  ent_coef={entropy_coef}')

    for ep in range(1, n_episodes + 1):
        routings, actions = sample_solution(policy, env, dd, greedy=False, temperature=temperature)
        sol   = Solution(env['demands'], routings)
        score, _ = evaluate(sol, env['net'], env['od_matrix'], env['settings'], env['yard_info'])

        advantage = baseline - score
        baseline  = baseline_decay * baseline + (1-baseline_decay) * score

        policy.train()
        log_p, entropy = compute_log_probs_entropy(policy, env, dd, actions)
        loss = -advantage * log_p - entropy_coef * entropy

        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        opt.step()

        if score < best_score:
            best_score = score
            best_state = deepcopy(policy.state_dict())

        log.append({'ep': ep, 'sample': score, 'best': best_score})

        if ep % print_every == 0:
            routings_g, _ = sample_solution(policy, env, dd, greedy=True)
            g, _ = evaluate(Solution(env['demands'], routings_g),
                            env['net'], env['od_matrix'], env['settings'], env['yard_info'])
            print(f'  ep {ep:4d} | sample={score:>14,.0f} | greedy={g:>14,.0f} | best={best_score:>14,.0f}')

    policy.load_state_dict(best_state)
    return best_score, log

## 5. SA Polishing

In [ ]:
def polish_with_sa(sol, env, sa_iters=80_000, verbose=True):
    """Run SA on the RL solution to squeeze out more improvement."""
    best_sol, best_obj, best_stats = simulated_annealing(
        sol, env['net'], env['od_matrix'], env['settings'], env['yard_info'],
        env['all_yards'], max_iter=sa_iters, verbose=verbose,
    )
    return best_sol, best_obj, best_stats

## 6. Full Pipeline: BC → RL → SA

In [ ]:
CASE_ORDER = [
    ('l1', 0.5), ('l1', 1.0), ('l1', 2.0),
    ('l2', 0.5), ('l2', 1.0), ('l2', 2.0),
    ('l3', 0.5), ('l3', 1.0), ('l3', 2.0),
]

# Hyperparams per layer
CONFIGS = {
    'l1': dict(bc_epochs=100, rl_eps=500,  sa_iters=100_000, temperature=1.5, entropy_coef=0.05),
    'l2': dict(bc_epochs=50,  rl_eps=200,  sa_iters=80_000,  temperature=1.3, entropy_coef=0.03),
    'l3': dict(bc_epochs=30,  rl_eps=100,  sa_iters=60_000,  temperature=1.2, entropy_coef=0.02),
}

solutions  = {}
train_logs = {}

In [ ]:
# ── L1 ────────────────────────────────────────────────────────────────────────
prev_policy_path = None

for mult in [0.5, 1.0, 2.0]:
    tag = f'l1_{mult}'
    cfg = CONFIGS['l1']
    print(f'\n{"="*60}\n  L1 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l1', mult)
    dd  = precompute(env)

    policy = make_policy()
    if prev_policy_path and os.path.exists(prev_policy_path):
        policy.load_state_dict(torch.load(prev_policy_path, map_location=DEVICE))
        print('  Loaded previous L1 weights')

    # Phase 1: BC
    init_sol, g0 = pretrain_bc(policy, env, dd,
                               n_epochs=cfg['bc_epochs'], print_every=25)

    # Phase 2: REINFORCE + Entropy
    rl_score, log = train_reinforce(policy, env, dd,
                                    n_episodes=cfg['rl_eps'],
                                    temperature=cfg['temperature'],
                                    entropy_coef=cfg['entropy_coef'])
    train_logs[tag] = log

    # RL best solution
    routings, _ = sample_solution(policy, env, dd, greedy=True)
    rl_sol = Solution(env['demands'], routings)

    # Phase 3: SA polishing
    print(f'\n  SA polishing...')
    best_sol, best_obj, best_stats = polish_with_sa(rl_sol, env,
                                                    sa_iters=cfg['sa_iters'])

    print(f'  Greedy={g0:,.0f} → RL={rl_score:,.0f} → SA={best_stats["stress_score"]:,.0f}')
    print(f'  blocks={best_stats["n_blocks"]}  elapsed={time.time()-t0:.1f}s')

    solutions[tag] = build_json(best_sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    path = f'saved_models/policy_l1_{mult}.pt'
    torch.save(policy.state_dict(), path)
    prev_policy_path = path

In [ ]:
# ── L2 (transfer from L1 1.0) ─────────────────────────────────────────────────
for mult in [0.5, 1.0, 2.0]:
    tag = f'l2_{mult}'
    cfg = CONFIGS['l2']
    print(f'\n{"="*60}\n  L2 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l2', mult)
    dd  = precompute(env)

    policy = make_policy()
    policy.load_state_dict(torch.load('saved_models/policy_l1_1.0.pt', map_location=DEVICE))
    print('  Transfer from L1 1.0')

    init_sol, g0 = pretrain_bc(policy, env, dd,
                               n_epochs=cfg['bc_epochs'], print_every=25)
    rl_score, log = train_reinforce(policy, env, dd,
                                    n_episodes=cfg['rl_eps'],
                                    temperature=cfg['temperature'],
                                    entropy_coef=cfg['entropy_coef'])
    train_logs[tag] = log

    routings, _ = sample_solution(policy, env, dd, greedy=True)
    rl_sol = Solution(env['demands'], routings)

    print(f'\n  SA polishing...')
    best_sol, best_obj, best_stats = polish_with_sa(rl_sol, env,
                                                    sa_iters=cfg['sa_iters'])

    print(f'  Greedy={g0:,.0f} → RL={rl_score:,.0f} → SA={best_stats["stress_score"]:,.0f}')
    print(f'  blocks={best_stats["n_blocks"]}  elapsed={time.time()-t0:.1f}s')

    solutions[tag] = build_json(best_sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save(policy.state_dict(), f'saved_models/policy_l2_{mult}.pt')

In [ ]:
# ── L3 (transfer from L2 1.0) ─────────────────────────────────────────────────
for mult in [0.5, 1.0, 2.0]:
    tag = f'l3_{mult}'
    cfg = CONFIGS['l3']
    print(f'\n{"="*60}\n  L3 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l3', mult)
    dd  = precompute(env)

    policy = make_policy()
    policy.load_state_dict(torch.load('saved_models/policy_l2_1.0.pt', map_location=DEVICE))
    print('  Transfer from L2 1.0')

    init_sol, g0 = pretrain_bc(policy, env, dd,
                               n_epochs=cfg['bc_epochs'], print_every=10)
    rl_score, log = train_reinforce(policy, env, dd,
                                    n_episodes=cfg['rl_eps'],
                                    temperature=cfg['temperature'],
                                    entropy_coef=cfg['entropy_coef'])
    train_logs[tag] = log

    routings, _ = sample_solution(policy, env, dd, greedy=True)
    rl_sol = Solution(env['demands'], routings)

    print(f'\n  SA polishing...')
    best_sol, best_obj, best_stats = polish_with_sa(rl_sol, env,
                                                    sa_iters=cfg['sa_iters'])

    print(f'  Greedy={g0:,.0f} → RL={rl_score:,.0f} → SA={best_stats["stress_score"]:,.0f}')
    print(f'  blocks={best_stats["n_blocks"]}  elapsed={time.time()-t0:.1f}s')

    solutions[tag] = build_json(best_sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save(policy.state_dict(), f'saved_models/policy_l3_{mult}.pt')

## 7. Generate submission.csv

In [ ]:
rows = [['ID', 'data']]
for case_id, (layer, mult) in enumerate(CASE_ORDER):
    tag = f'{layer}_{mult}'
    sol = solutions.get(tag)
    if sol is None:
        print(f'[ID {case_id}] {tag} MISSING')
        rows.append([case_id, '{}'])
        continue
    data_str = json.dumps(sol, separators=(',', ':'))
    n_blocks = len(sol['outputs']['1 Block Design'])
    n_seqs   = len(sol['outputs']['2 Blocking Sequence'])
    print(f'[ID {case_id}] {tag}  blocks={n_blocks}  seqs={n_seqs}')
    rows.append([case_id, data_str])

with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(rows)

print(f'\nDone. {sum(len(r[1]) for r in rows[1:])/1e6:.1f} MB')

In [ ]:
from google.colab import files
files.download('submission.csv')

## 8. Training Curves (paper)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (layer, mult) in zip(axes, [('l1',1.0),('l2',1.0),('l3',1.0)]):
    tag = f'{layer}_{mult}'
    log = train_logs.get(tag, [])
    if not log: continue
    ax.plot([d['ep'] for d in log], [d['best'] for d in log])
    ax.set_title(f'{layer.upper()} ×{mult} (REINFORCE+Entropy)')
    ax.set_xlabel('Episode'); ax.set_ylabel('Best Stress Score')
    ax.grid(True, alpha=0.3)

plt.suptitle('RL Training Curves — Best Score per Episode')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()